# GeoGram: Klasifikace sg/pl — Wikipedia (všechny -ice obce)

Notebook klasifikuje gramatické číslo obcí přes Czech Wikipedia intro text.

**Výstup:** `data/processed/ice_grammar_wiki.csv`

### Jak funguje klasifikace
| krok | logika |
|---|---|
| 1. Exact title | `Mohelnice` — pokud stránka existuje a není rozcestník |
| 2. Okres qualifier | `Bystřice (okres Benešov)` — pro ambiguitní jména |
| 3. Search fallback | `Bystřice obec Benešov` přes search API |
| 4. Extrakce čísla | regex na `X jsou…` / `X je…` v první větě; keyword `pomnožné` |

### Kategorie výsledků
| hodnota | význam |
|---|---|
| `singular` | verb `je` za názvem → singulár |
| `plural` | verb `jsou` za názvem → plurál |
| `unknown` | stránka nalezena, ale text bez jasného vzoru |
| `not_found` | stránka nenalezena ani po disambiguaci |
| `error` | síťová / HTTP chyba |

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from geogram.wikipedia import (
    fetch_municipality_number,
    SINGULAR, PLURAL, BOTH, UNKNOWN, NOT_FOUND,
)

PROCESSED_DIR = project_root / 'data' / 'processed'
INPUT_CSV   = PROCESSED_DIR / 'municipalities_ice_integrated.csv'
CHECKPOINT  = PROCESSED_DIR / 'wiki_checkpoint.csv'
OUTPUT_CSV  = PROCESSED_DIR / 'ice_grammar_wiki.csv'

# Wikipedia API je benevolentnější než ÚJČ — 0.5s stačí
SLEEP_BETWEEN_REQUESTS = 0.5
CHECKPOINT_EVERY       = 100

# Re-procesovat tyto kategorie z předchozího běhu
REPROCESS_CATEGORIES = {'unknown', 'error', 'not_found'}

print(f'Input:      {INPUT_CSV}')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Output:     {OUTPUT_CSV}')

Input:      C:\Users\dobes\Documents\UniversityCodingProject\NewFunnyProjects\GeoGram_sufix-ice\data\processed\municipalities_ice_integrated.csv
Checkpoint: C:\Users\dobes\Documents\UniversityCodingProject\NewFunnyProjects\GeoGram_sufix-ice\data\processed\wiki_checkpoint.csv
Output:     C:\Users\dobes\Documents\UniversityCodingProject\NewFunnyProjects\GeoGram_sufix-ice\data\processed\ice_grammar_wiki.csv


## 2. Načtení dat a resumování

In [2]:
df = pd.read_csv(INPUT_CSV)
print(f'Načteno {len(df):,} obcí')

if CHECKPOINT.exists():
    done = pd.read_csv(CHECKPOINT)
    done_codes = set(done[~done['wiki_number'].isin(REPROCESS_CATEGORIES)]['code'])
    print(f'Checkpoint: {len(done):,} záznamů, {len(done_codes):,} definitivních.')
elif OUTPUT_CSV.exists() and REPROCESS_CATEGORIES:
    prev = pd.read_csv(OUTPUT_CSV)[['code', 'name', 'wiki_number', 'wiki_status']]
    done = prev[~prev['wiki_number'].isin(REPROCESS_CATEGORIES)]
    done_codes = set(done['code'])
    print(f'Předchozí výstup: {len(prev):,} záznamů, re-procesuju {len(prev)-len(done):,}.')
else:
    done = pd.DataFrame(columns=['code', 'name', 'wiki_number', 'wiki_status'])
    done_codes = set()
    print('Začínám od začátku.')

todo = df[~df['code'].isin(done_codes)].reset_index(drop=True)
print(f'Zbývá: {len(todo):,} obcí  (~{len(todo)*SLEEP_BETWEEN_REQUESTS/60:.0f} min)')

Načteno 1,806 obcí
Začínám od začátku.
Zbývá: 1,806 obcí  (~15 min)


## 3. Klasifikace

In [3]:
results = done.to_dict('records')
errors = []

pbar = tqdm(todo.iterrows(), total=len(todo), unit='obec',
            desc='Wiki klasifikace', dynamic_ncols=True)

for i, row in pbar:
    name         = row['name']
    code         = row['code']
    district     = row.get('district_name', '')
    region       = row.get('region_name', '')

    try:
        number, status = fetch_municipality_number(name, district, region)
    except Exception as exc:
        number, status = 'error', 'error'
        errors.append({'code': code, 'name': name, 'error': str(exc)})

    results.append({'code': code, 'name': name,
                    'wiki_number': number, 'wiki_status': status})

    pbar.set_postfix({'poslední': name[:18], 'výsledek': number,
                      'status': status, 'chyby': len(errors)})

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False, encoding='utf-8')

    time.sleep(SLEEP_BETWEEN_REQUESTS)

pbar.close()
print(f'\nHotovo. Chyby: {len(errors)}')
if errors:
    for e in errors:
        print(f"  {e['name']}: {e['error']}")

Wiki klasifikace:   0%|          | 0/1806 [00:00<?, ?obec/s]


Hotovo. Chyby: 0


## 4. Uložení výsledků

In [4]:
df_results = pd.DataFrame(results)
df_final = df.merge(
    df_results[['code', 'wiki_number', 'wiki_status']],
    on='code', how='left'
)

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
if CHECKPOINT.exists():
    CHECKPOINT.unlink()

print(f'Uloženo {len(df_final):,} řádků → {OUTPUT_CSV}')
df_final[['name', 'region_name', 'population_total', 'wiki_number', 'wiki_status']].head(10)

Uloženo 1,806 řádků → C:\Users\dobes\Documents\UniversityCodingProject\NewFunnyProjects\GeoGram_sufix-ice\data\processed\ice_grammar_wiki.csv


,name,region_name,population_total,wiki_number,wiki_status
0,České Budějovice,Jihočeský kraj,97231.0,plural,found
1,Pardubice,Pardubický kraj,92319.0,plural,found
2,Teplice,Ústecký kraj,50912.0,plural,found
3,Litoměřice,Ústecký kraj,22767.0,plural,found
4,Strakonice,Jihočeský kraj,22355.0,plural,found
5,Kopřivnice,Moravskoslezský kraj,21374.0,singular,found
6,Hranice,Olomoucký kraj,17969.0,singular,found
7,Otrokovice,Zlínský kraj,17401.0,plural,found
8,Neratovice,Středočeský kraj,16360.0,plural,found
9,Milovice,Středočeský kraj,14270.0,plural,found


## 5. Statistiky a pokrytí

In [5]:
counts = df_final['wiki_number'].value_counts()
total = len(df_final)
classified = df_final['wiki_number'].isin([SINGULAR, PLURAL, BOTH]).sum()

print('=== Distribuce gramatického čísla (Wikipedia) ===')
for label, n in counts.items():
    print(f'  {label:<12}: {n:>5}  ({n/total*100:.1f} %)')
print(f'\nKlasifikováno:  {classified:,} ({classified/total*100:.1f} %)')
print(f'Bez stránky:    {counts.get(NOT_FOUND, 0):,}')

=== Distribuce gramatického čísla (Wikipedia) ===
  plural      :  1389  (76.9 %)
  singular    :   227  (12.6 %)
  unknown     :   190  (10.5 %)

Klasifikováno:  1,616 (89.5 %)
Bez stránky:    0


In [6]:
# Jak často bylo potřeba disambiguovat?
status_counts = df_final['wiki_status'].value_counts()
print('=== Způsob nalezení (resolution strategy) ===')
for label, n in status_counts.items():
    print(f'  {label:<25}: {n:>5}  ({n/total*100:.1f} %)')

=== Způsob nalezení (resolution strategy) ===
  found                    :  1331  (73.7 %)
  resolved_with_okres      :   475  (26.3 %)


In [7]:
# Sg/pl po krajích
df_known = df_final[df_final['wiki_number'].isin([SINGULAR, PLURAL, BOTH])]

pivot = (
    df_known
    .groupby(['region_name', 'wiki_number'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .sort_values('total', ascending=False)
)
print('=== Sg/pl po krajích (Wikipedia) ===')
print(pivot.to_string())

=== Sg/pl po krajích (Wikipedia) ===
wiki_number           plural  singular  total
region_name                                  
Středočeský kraj         243        38    281
Jihomoravský kraj        253        17    270
Jihočeský kraj           144        26    170
Kraj Vysočina            132        23    155
Královéhradecký kraj      87        33    120
Plzeňský kraj            105        14    119
Olomoucký kraj            91        18    109
Pardubický kraj           88        13    101
Zlínský kraj              88        10     98
Moravskoslezský kraj      68         9     77
Ústecký kraj              57        11     68
Liberecký kraj            20        11     31
Karlovarský kraj          13         4     17


In [8]:
# Top příklady z každé kategorie
for label in [SINGULAR, PLURAL, BOTH, UNKNOWN, NOT_FOUND, 'error']:
    subset = df_final[df_final['wiki_number'] == label]
    if subset.empty:
        continue
    top = subset.nlargest(5, 'population_total')[['name', 'region_name', 'population_total', 'wiki_status']]
    print(f'\n--- {label.upper()} (top 5 podle populace) ---')
    print(top.to_string(index=False))


--- SINGULAR (top 5 podle populace) ---
      name          region_name  population_total         wiki_status
Kopřivnice Moravskoslezský kraj           21374.0               found
   Hranice       Olomoucký kraj           17969.0               found
    Sušice        Plzeňský kraj           10740.0               found
  Jesenice     Středočeský kraj           10460.0 resolved_with_okres
 Mohelnice       Olomoucký kraj            9782.0               found

--- PLURAL (top 5 podle populace) ---
            name     region_name  population_total wiki_status
České Budějovice  Jihočeský kraj           97231.0       found
       Pardubice Pardubický kraj           92319.0       found
         Teplice    Ústecký kraj           50912.0       found
      Litoměřice    Ústecký kraj           22767.0       found
      Strakonice  Jihočeský kraj           22355.0       found

--- UNKNOWN (top 5 podle populace) ---
          name          region_name  population_total wiki_status
     Palkovice M